# 01 – Citi Bike-data automatisch downloaden en samenvatten

| | |
|---|---|
| **Auteur(s)** | Kobe Vandenberk |
| **Wat** | Alle ritten van 2024 automatisch van de officiële bron halen, controleren en in één passage samenvatten tot *vertrekken per station per uur*. Daarnaast het uurlijkse weer in NYC ophalen. |
| **Input** | `https://s3.amazonaws.com/tripdata/2024MM-citibike-tripdata.zip` (12 maanden), Open-Meteo archive API |
| **Output** | `data/raw/*.zip` + `manifest.json`, `data/station_hour.parquet`, `data/stations.parquet`, `data/audit.json`, `data/trips_sample.parquet`, `data/weather_2024.parquet` |
| **GenAI** | Code en tekst opgesteld met hulp van Claude; keuzes en resultaten nagekeken door Kobe. |

## Onderzoeksvraag van deze reeks notebooks
**Hoeveel fietsen vertrekken er in een bepaalde zone van New York in een bepaald uur?**

Dat is de vraag die Citi Bike elke dag moet beantwoorden om fietsen te *herverdelen*: lege stations in de spits betekenen dat mensen de auto, taxi of metro nemen. Een goede voorspelling van de vraag helpt dus rechtstreeks het thema **Going green**.

Mijn teamgenoten bekeken al de *ritduur* (Brent) en *member vs. casual* (Dimi). Vraagvoorspelling is een ander soort probleem: een **tijdreeks van tellingen** (zie AWS-hoofdstuk 4: seizoenen, cyclische patronen, backtesting).

## Waarom niet gewoon alle ritten in één DataFrame?
2024 telt meer dan **44 miljoen** ritten (± 7 GB aan CSV's). Dat past niet comfortabel in het geheugen van een laptop. Voor onze vraag hebben we de individuele ritten ook niet nodig: we tellen vertrekken. Daarom lezen we elke maand **in stukken van 500.000 rijen** en houden we per stuk enkel bij:
- het aantal vertrekken per station per uur (+ hoeveel daarvan members / e-bikes waren);
- een stationslijst (naam, coördinaten);
- een **audit**: hoeveel rijen, ontbrekende waarden, duplicaten en wat we weglieten;
- een **ruwe steekproef van 1%** van alle ritten, zodat we in de EDA ook naar individuele ritten kunnen kijken.

Alle verwerkingscode staat in `citibike.py`, zodat alle notebooks en `train.py` (het script dat het model voor de API schrijft) exact hetzelfde doen.

## 0. Imports

In [1]:
import json

import pandas as pd

import citibike as cb

pd.set_option("display.max_columns", 30)
print("Maanden:", cb.MONTHS[0], "...", cb.MONTHS[-1])
print("Data-map:", cb.DATA_DIR)

Maanden: 202401 ... 202412
Data-map: C:\Users\kobev\Documents\semester 1\3APPAI - Deep Learning\CloudAI-Challenge\citi_bike\Kobe\data


## 1. Downloaden (met controle)
De bestanden staan in een publieke S3-bucket. Per maand:
1. de verwachte grootte opvragen (`HEAD`-request);
2. downloaden naar een `.part`-bestand en pas hernoemen als de grootte klopt (een afgebroken download wordt zo nooit voor een geldige zip aangezien);
3. een **SHA-256 checksum** berekenen en alles in `manifest.json` bewaren. Zo kan iedereen controleren dat hij exact dezelfde data gebruikt.

Bestanden die al volledig aanwezig zijn, worden niet opnieuw gedownload.

In [2]:
manifest = cb.download_months()
manifest[["month", "bytes", "status", "csv_parts"]].assign(MB=lambda d: (d["bytes"] / 1e6).round()).drop(columns="bytes")

202401: al aanwezig    369 MB, 2 CSV-delen


202402: al aanwezig    415 MB, 3 CSV-delen


202403: al aanwezig    521 MB, 3 CSV-delen


202404: al aanwezig    630 MB, 4 CSV-delen


202405: al aanwezig    809 MB, 5 CSV-delen


...

202409: al aanwezig    977 MB, 5 CSV-delen


202410: al aanwezig   1007 MB, 6 CSV-delen


202411: al aanwezig    724 MB, 4 CSV-delen


202412: al aanwezig    451 MB, 3 CSV-delen


,month,status,csv_parts,MB
0,202401,al aanwezig,"[202401-citibike-tripdata_1.csv, 202401-citibi...",369.0
1,202402,al aanwezig,"[202402-citibike-tripdata_1.csv, 202402-citibi...",415.0
2,202403,al aanwezig,"[202403-citibike-tripdata_1.csv, 202403-citibi...",521.0
3,202404,al aanwezig,"[202404-citibike-tripdata_1.csv, 202404-citibi...",630.0
4,202405,al aanwezig,"[202405-citibike-tripdata_1.csv, 202405-citibi...",809.0
5,202406,al aanwezig,"[202406-citibike-tripdata_1.csv, 202406-citibi...",936.0
6,202407,al aanwezig,"[202407-citibike-tripdata_1.csv, 202407-citibi...",923.0
7,202408,al aanwezig,"[202408-citibike-tripdata_1.csv, 202408-citibi...",900.0
8,202409,al aanwezig,"[202409-citibike-tripdata_1.csv, 202409-citibi...",977.0
9,202410,al aanwezig,"[202410-citibike-tripdata_1.csv, 202410-citibi...",1007.0


**Vaststelling:** vanaf 2024 bevat elke maand-zip meerdere CSV-delen (een deel heeft hoogstens ± 1 miljoen rijen). We lezen ze allemaal.

## 2. Eén passage door alle maanden
`aggregate_all` leest per maand alle CSV-delen in stukken. Per stuk:
- zetten we de tekst om naar datums/getallen (`parse_chunk`);
- herstellen we station-id's die als getal werden opgeslagen (`3423.1` → `3423.10`, zie 02);
- houden we alleen **vertrekken met een gekende starttijd, startstation en coördinaten binnen NYC** (`departure_mask`);
- tellen we per station per uur.

De ritduur gebruiken we hier bewust **niet** als filter: een rit van 3 uur is nog altijd een fiets die vertrok. Of extreme ritten iets betekenen, bekijken we in 02.

In [3]:
result = cb.aggregate_all()

202401:  1,888,085 rijen ->  1,886,925 vertrekken (21 s)


202402:  2,121,501 rijen ->  2,119,635 vertrekken (21 s)


202403:  2,663,295 rijen ->  2,660,499 vertrekken (34 s)


202404:  3,217,063 rijen ->  3,214,557 vertrekken (27 s)


202405:  4,133,961 rijen ->  4,132,274 vertrekken (34 s)


...

202408:  4,603,575 rijen ->  4,600,613 vertrekken (25 s)


202409:  4,997,898 rijen ->  4,993,438 vertrekken (26 s)


202410:  5,150,054 rijen ->  5,145,472 vertrekken (27 s)


202411:  3,710,134 rijen ->  3,707,833 vertrekken (19 s)


202412:  2,311,171 rijen ->  2,310,051 vertrekken (15 s)


## 3. Wat hebben we nu?

In [4]:
audit = pd.json_normalize(result["audit"]).set_index("month")
audit[["rows", "departures", "no_start_station", "outside_nyc", "station_id_fixed",
       "duplicate_ride_ids", "duration_over_24h", "duration_negative"]]

,rows,departures,no_start_station,outside_nyc,station_id_fixed,duplicate_ride_ids,duration_over_24h,duration_negative
month,,,,,,,,
202401,1888085,1886925,1160,0,58970,0,607,0
202402,2121501,2119635,1866,0,20268,0,660,0
202403,2663295,2660499,2796,0,50037,0,841,0
202404,3217063,3214557,2506,0,48881,0,908,0
202405,4133961,4132274,1687,0,39452,0,1076,0
202406,4783576,4781821,1755,0,41115,0,1387,0
202407,4722896,4720256,2640,0,6520,0,1294,0
202408,4603575,4600613,2962,0,11860,0,1198,0
202409,4997898,4993438,4460,0,14692,0,1123,0


In [5]:
totals = audit[["rows", "departures"]].sum()
print(f"Ruwe rijen 2024:      {totals['rows']:>12,}")
print(f"Geldige vertrekken:   {totals['departures']:>12,}  ({totals['departures'] / totals['rows']:.3%})")
print(f"Station x uur-rijen:  {len(result['station_hour']):>12,}")
print(f"Stations:             {len(result['stations']):>12,}")

Ruwe rijen 2024:        44,303,209
Geldige vertrekken:     44,273,374  (99.933%)
Station x uur-rijen:     9,535,664
Stations:                    2,280


**Interpretatie:** 44,3 miljoen ritten in 2024, waarvan **99,93%** bruikbaar is als vertrek. Wat wegvalt, zijn vooral ritten zonder startstation (en dus zonder coördinaten). De samenvatting is véél kleiner dan de ruwe data: van 44 miljoen ritten naar 9,5 miljoen station×uur-tellingen (± 38 MB parquet).

## 4. Weer ophalen
Het weer is de belangrijkste factor buiten de kalender: wie fietst er in de stromende regen? We halen per uur temperatuur, neerslag, wind, sneeuw en luchtvochtigheid op bij **Open-Meteo** (gratis archief van weermodellen, geen API-sleutel nodig) voor één punt in het midden van NYC.

Belangrijk: we vragen het weer in **lokale tijd** (`America/New_York`), want ook de ritdata staat in lokale tijd. Door de zomertijd ontbreekt in maart één uur en komt in november één uur dubbel voor; `download_weather` ontdubbelt en vult dat ene uur op met interpolatie.

In [6]:
weather = cb.download_weather()
print(weather.shape)
weather.describe().round(1)

(8784, 6)


,time,temperature_c,precipitation_mm,wind_kmh,snowfall_cm,humidity_pct
count,8784,8784.0,8784.0,8784.0,8784.0,8784.0
mean,2024-07-01 23:30:00,13.1,0.1,12.8,0.0,69.8
min,2024-01-01 00:00:00,-14.7,0.0,0.2,0.0,13.0
25%,2024-04-01 11:45:00,5.2,0.0,8.1,0.0,56.0
50%,2024-07-01 23:30:00,13.4,0.0,11.6,0.0,72.0
75%,2024-10-01 11:15:00,21.2,0.0,16.5,0.0,85.0
max,2024-12-31 23:00:00,35.5,18.6,60.6,2.2,100.0
std,NaN,9.8,0.8,6.6,0.1,18.4


**Controle:** 366 dagen × 24 uur = 8.784 uren (2024 is een schrikkeljaar), zonder ontbrekende waarden.

In [7]:
assert len(weather) == 366 * 24
assert weather.drop(columns="time").notna().all().all()
print("Weerdata compleet.")

Weerdata compleet.


## Samenvatting
- Alle data wordt **met code** opgehaald en gecontroleerd (grootte + checksum in `manifest.json`); niets gebeurt manueel.
- We bewaren enkel wat we nodig hebben: vertrekken per station per uur, de stationslijst, een audit en een steekproef van 1% voor de EDA.
- Volgende stap: `02_eda_cleaning.ipynb`.